# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Artasam/Machine-Learning/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook is the data contract for **Lane 2 — Refresh / Content Opportunity Scoring**.
It answers five plain-English questions about the data, proves three facts with real DuckDB
queries on the warehouse release, builds a five-feature frame, demonstrates the classic
leakage trap, and states one named limitation.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load
> `writing-data-contracts` + `flyrank/flyrank-data` for this task.

In [5]:
# Install dependencies (runs in < 30 s on Colab; skip if already installed locally)
%pip -q install duckdb huggingface_hub requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os, getpass

# Try Colab Secrets first (🔑 key panel → add a secret named HF_TOKEN), then env var, then prompt.
# NEVER paste your token into a code cell — this repo is public.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

print('Token loaded:', 'YES ✅' if HF_TOKEN else 'NO ❌ — check your Colab Secrets panel')

Token loaded: YES ✅


In [7]:
# =============================================================================
# PRE-FLIGHT CHECK  — Run this BEFORE any DuckDB query.
# =============================================================================
# The "ZSTD Decompression failure" error means DuckDB received an HTML 401/403
# page instead of a real Parquet file, then tried (and failed) to unzip it.
# This cell checks BOTH root causes with a clear, actionable error message.
# =============================================================================
import requests

WHOAMI_URL = 'https://huggingface.co/api/whoami-v2'
GATE_URL   = 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse'
headers    = {'Authorization': f'Bearer {HF_TOKEN}'}

# --- Step 1: Is the token valid? ---
r_who = requests.get(WHOAMI_URL, headers=headers, timeout=10)
if r_who.status_code == 200:
    username = r_who.json().get('name', 'unknown')
    print(f'✅ Token valid. HF account: {username}')
else:
    raise RuntimeError(
        f'❌ Token rejected (HTTP {r_who.status_code}).\n'
        'Fix: go to https://huggingface.co/settings/tokens\n'
        '     → Create new token → choose plain "Read" type (NOT Fine-grained)\n'
        '     → copy it → paste in Colab Secrets panel (🔑) as HF_TOKEN\n'
        '     → re-run from the top.'
    )

# --- Step 2: Is the gate accepted? ---
r_gate = requests.get(GATE_URL, headers=headers, timeout=10)
if r_gate.status_code == 403:
    raise RuntimeError(
        '❌ Gate not accepted (HTTP 403).\n'
        'Fix: open https://huggingface.co/datasets/FlyRank/internship-warehouse\n'
        '     → click "Request access" → fill affiliation as "FlyRank ML Internship 2026"\n'
        '     → tick the data-use terms → click Agree. Approval is INSTANT.\n'
        '     Then re-run this cell.\n'
        'NOTE: The token MUST be created AFTER you accept the gate, not before.'
    )
elif r_gate.status_code == 200:
    print('✅ Gate accepted — you have access to FlyRank/internship-warehouse.')
else:
    print(f'⚠️  Unexpected status {r_gate.status_code} — proceeding, but watch for errors.')

print('\nAll pre-flight checks passed. Safe to run DuckDB queries.')

✅ Token valid. HF account: Artasam-Khan
✅ Gate accepted — you have access to FlyRank/internship-warehouse.

All pre-flight checks passed. Safe to run DuckDB queries.


In [8]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# We query ONE partition (month=2026-03) for speed.
# NEVER use the _sample table for experiments — it is the last month (June 2026),
# which is the answer key for any future-window label.
FACT_MARCH  = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Smoke-test on the SMALLEST table (dim_clients, 104 rows) to confirm DuckDB auth works
# BEFORE attempting the large fact table. Fast and definitive.
try:
    n = con.sql(f"SELECT COUNT(*) FROM {DIM_CLIENTS}").fetchone()[0]
    print(f'✅ DuckDB auth confirmed. dim_clients has {n} rows.')
    print('Ready to query fact_content_daily_performance/month=2026-03')
except Exception as e:
    raise RuntimeError(
        f'DuckDB cannot read the warehouse: {e}\n'
        'Most likely cause: token invalid or gate not accepted.\n'
        'Run the pre-flight cell above and fix the error it reports.'
    ) from e

✅ DuckDB auth confirmed. dim_clients has 104 rows.
Ready to query fact_content_daily_performance/month=2026-03


---

## 1. Unit of analysis + time window

**Five plain-words contract answers:**

1. **What one row means for my lane:**  
   One row = one **page** (`content_hash_id`), belonging to one **client** (`client_hash_id`),  
   described by its aggregated search signals over a 31-day window (March 2026).  
   Every row represents one page's performance snapshot — not a daily record.

2. **Which table I'll use:**  
   `fact_content_daily_performance`, partitioned by month, queried one month at a time  
   (`month=2026-03`). I aggregate the daily rows into one per page using `GROUP BY`.

3. **Which time window:**  
   The **feature window** is March 2026 (`2026-03-01` → `2026-03-31`).  
   The **proxy label** compares the last 15 days vs the first 15 days of that window,
   so it stays inside the same month and avoids looking forward into the future.

4. **What I predict (label or proxy):**  
   `is_declining_proxy` = 1 if the page lost more than 20% of impressions in the second  
   half of March vs the first half. This is a **proxy label** (a defined rule on  
   current-window data), not a genuine forward-looking outcome — stated honestly.

5. **One thing I deliberately exclude:**  
   All GA4-derived columns (`ga4_sessions`, `ga4_pageviews`, etc.) when  
   `ga4_data_available IS NOT TRUE`. Rows before a client's GA4 start date have  
   zero-filled GA4 columns. Treating those zeros as "no engagement" is wrong —  
   they mean "no tracking yet". I filter with `ga4_data_available IS TRUE` throughout.

In [9]:
# --- Query 1 of 3: Grain verification ---
# Prove that (report_date, client_hash_id, content_hash_id) is the real grain.
# Zero rows returned = grain holds perfectly (no duplicates).

grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT_MARCH}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""").df()

print("Grain check (should be 0 rows if grain holds):")
print(f"Duplicate combinations found: {len(grain_check)}")
print(grain_check)

InvalidInputException: Invalid Input Error: Failed to read file "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet": ZSTD Decompression failure

---

## 2. Fields: feature / label / context / excluded

Every column I plan to touch is in exactly one bucket:

| Column | Bucket | Reason |
|---|---|---|
| `gsc_impressions` (sum over March) | **Feature** | Total search visibility — knowable at end of month |
| `gsc_clicks` (sum over March) | **Feature** | Total clicks earned — measurable, available at decision time |
| `gsc_avg_position` (avg over days with data) | **Feature** | Average search rank — directly measurable |
| `days_with_impressions` (count of active days) | **Feature** | Consistency signal — observable, not a forecast |
| `momentum_ratio` (imp_last15 / imp_prev15) | **Feature** | Trend within the month — both halves are in the past at decision time |
| `is_declining_proxy` | **Label (proxy)** | Derived from `imp_last15 < 0.8 × imp_prev15`. The thing we predict. NEVER a feature |
| `content_hash_id` | **Context** | Page identifier — for grouping and joining only |
| `client_hash_id` | **Context** | Client identifier — used for client-holdout splits only |
| `report_date` | **Context** | Used for window construction only, not a feature |
| `ga4_sessions`, `ga4_pageviews`, etc. | **Excluded** | Only available when `ga4_data_available IS TRUE`. Including zero-filled rows injects systematic noise |
| `ga4_data_available` | **Excluded** | Used as a filter flag only — a page's tracking start date is not a search signal |

In [ ]:
# Inspect the actual columns present in the March fact table
# so our contract matches what's really there.
cols = con.sql(f"DESCRIBE SELECT * FROM {FACT_MARCH} LIMIT 1").df()
print("All columns in fact_content_daily_performance (march partition):")
print(cols[['column_name', 'column_type']].to_string())

---

## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim above gets a query cell. A contract line without a query is a guess.

In [ ]:
# --- Query 2 of 3: Row count and date span ---
# Confirms: how many raw daily rows are in March 2026, and what is the real date range?

span = con.sql(f"""
    SELECT
        COUNT(*)                                AS total_rows,
        COUNT(DISTINCT content_hash_id)         AS distinct_pages,
        COUNT(DISTINCT client_hash_id)          AS distinct_clients,
        MIN(report_date)                        AS earliest_date,
        MAX(report_date)                        AS latest_date
    FROM {FACT_MARCH}
""").df()

print("Slice row count and date span (month=2026-03):")
print(span.to_string())

In [ ]:
# --- Query 3 of 3: Availability filter (IS TRUE rule) ---
# ga4_data_available can be TRUE, FALSE, or NULL.
# The skill file warns: always write `IS TRUE` because:
#   - `= TRUE`    misses NULLs (NULL = TRUE is NULL, not TRUE)
#   - `NOT FALSE` keeps NULLs in (NULL is neither TRUE nor FALSE)
# This query shows how many rows survive the correct IS TRUE filter.

avail = con.sql(f"""
    SELECT
        COUNT(*)                                            AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable_or_null,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*),
        1)                                                  AS pct_ga4_available
    FROM {FACT_MARCH}
""").df()

print("Availability check — rows with real GA4 data (ga4_data_available IS TRUE):")
print(avail.to_string())
print()
print("Rows where ga4_data_available IS NOT TRUE are EXCLUDED from any GA4 features.")
print("These zeros mean 'no tracking yet', not 'no engagement'.")

---

## 4. Five features + the leakage trap

### The five features I'll build

Each feature gets one line: what it is, and why it is **knowable at the decision moment**.

| # | Feature | Available when? |
|---|---|---|
| 1 | `total_impressions` — sum of `gsc_impressions` over March | Knowable at end of March: it's a count of what happened, not a prediction |
| 2 | `total_clicks` — sum of `gsc_clicks` over March | Knowable at end of March for the same reason |
| 3 | `avg_position` — average `gsc_avg_position` over days with data | Knowable at end of March; position is measured daily by Search Console |
| 4 | `days_active` — count of days where `gsc_impressions > 0` | Knowable at end of March; consistency signal, not a forecast |
| 5 | `momentum_ratio` — `imp_last15 / (imp_prev15 + 1)` | Both halves of March are in the past at decision time. +1 avoids division by zero |

**The proxy label** `is_declining_proxy` = 1 if `imp_last15 < 0.8 × imp_prev15`  
(a >20% drop in the second half of March vs the first half).  
This is the target — **never a feature**.

In [ ]:
import pandas as pd
import numpy as np

# Build the five-feature frame: one row per page, aggregated from daily rows.
# We split March into two halves to build the momentum feature and the proxy label.

feature_frame = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,

        -- Feature 1: total search impressions over March
        SUM(gsc_impressions)                                            AS total_impressions,

        -- Feature 2: total clicks over March
        SUM(gsc_clicks)                                                 AS total_clicks,

        -- Feature 3: average position (only over days that had position data)
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)  AS avg_position,

        -- Feature 4: how many days had any impressions (consistency)
        SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END)           AS days_active,

        -- Split for Feature 5 (momentum) and the proxy label
        SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_prev15,
        SUM(CASE WHEN report_date >  '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_last15

    FROM {FACT_MARCH}
    WHERE gsc_impressions > 0          -- pages with any search visibility
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) >= 50  -- minimum signal threshold
""").df()

# Feature 5: momentum ratio (computed in pandas after aggregation)
feature_frame['momentum_ratio'] = (
    feature_frame['imp_last15'] / (feature_frame['imp_prev15'] + 1)
)

# Proxy label: >20% drop in second half vs first half of March
feature_frame['is_declining_proxy'] = (
    feature_frame['imp_last15'] < 0.8 * feature_frame['imp_prev15']
).astype(int)

print(f"Feature frame: {len(feature_frame):,} pages x {len(feature_frame.columns)} columns")
print(f"Proxy label base rate: {feature_frame['is_declining_proxy'].mean():.1%} declining")
print()
print(feature_frame[['total_impressions', 'total_clicks', 'avg_position',
                       'days_active', 'momentum_ratio', 'is_declining_proxy']].describe().round(2))

In [ ]:
# Show the first 5 rows so the contract is concrete.
print("Sample of the feature frame (5 rows):")
feature_frame[['content_hash_id', 'client_hash_id',
               'total_impressions', 'total_clicks', 'avg_position',
               'days_active', 'momentum_ratio', 'is_declining_proxy']].head()

### The leakage trap — performed and removed

The trap: add a column that is directly computed from the label and watch precision jump
toward 1.0. The correct response is to recognise what happened and delete the column.

Here I add `imp_last15` itself as a "feature". Since `is_declining_proxy` is literally
defined as `imp_last15 < 0.8 × imp_prev15`, giving the model `imp_last15` hands it the
answer — that is leakage.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

# -----------------------------------------------------------------------
# STEP A: Honest model — 5 safe features only
# -----------------------------------------------------------------------
HONEST_FEATURES = ['total_impressions', 'total_clicks', 'avg_position',
                   'days_active', 'momentum_ratio']

df_clean = feature_frame.dropna(subset=HONEST_FEATURES).copy()
X_honest = df_clean[HONEST_FEATURES]
y        = df_clean['is_declining_proxy']

X_tr, X_te, y_tr, y_te = train_test_split(
    X_honest, y, test_size=0.25, random_state=42, stratify=y
)
model_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_honest.fit(X_tr, y_tr)
prec_honest = precision_score(y_te, model_honest.predict(X_te))

print(f"Honest model Precision  (5 safe features):    {prec_honest:.3f}")

# -----------------------------------------------------------------------
# STEP B: Leaked model — add imp_last15 (part of the label formula)
# -----------------------------------------------------------------------
LEAKED_FEATURES = HONEST_FEATURES + ['imp_last15']   # <<< THE TRAP

X_leaked = df_clean[LEAKED_FEATURES]
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(
    X_leaked, y, test_size=0.25, random_state=42, stratify=y
)
model_leaked = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_leaked.fit(X_tr_l, y_tr_l)
prec_leaked = precision_score(y_te_l, model_leaked.predict(X_te_l))

print(f"Leaked model Precision   (+ imp_last15):      {prec_leaked:.3f}  ← suspiciously high")
print()
print("EXPLANATION:")
print("  imp_last15 is literally one side of the label formula:")
print("    is_declining_proxy = (imp_last15 < 0.8 * imp_prev15)")
print("  The model is given the answer to predict the answer.")
print("  In the real world we would NOT have imp_last15 at prediction time.")
print()
print("ACTION: imp_last15 removed from all further modeling. Only 5 honest features kept.")

del model_leaked  # deleted — only model_honest is valid going forward

---

## 5. Data limits — one named limitation of this slice

**Named limitation: unbalanced client panel — proxy label quality varies by client history depth.**

The warehouse is an **unbalanced panel**: different clients started tracking on different dates.
Some clients have 17 months of daily data; others have 3. In March 2026, a page from a
new client might have only a few weeks of history, making the `imp_prev15 / imp_last15`
comparison meaningless (both halves could be near-zero from incomplete tracking).

The `momentum_ratio` feature and the `is_declining_proxy` label are both built from the
two halves of March. For a client whose `ga4_data_start` falls inside March, the first half
will look artificially depressed, causing the model to incorrectly flag those pages as
"declining" when they were simply not yet tracked.

**The mitigation I apply:** filter on `gsc_impressions > 0` and apply a minimum
`gsc_impressions >= 50` threshold (see the feature frame). This removes most incomplete
rows, but does not fully solve the unbalanced-panel problem — a proper fix requires checking
`dim_clients.gsc_data_start` per client before computing any window, which is a Week 4+ task.

**This data can never tell me:** whether refreshing a page causes recovery. The data shows
what happened in the observation window; no controlled experiment was run.

In [ ]:
# Show the unbalanced panel concretely: per-client data start dates from dim_clients.
# This backs the limitation claim above with real numbers.

client_history = con.sql(f"""
    SELECT
        client_hash_id,
        gsc_data_start,
        ga4_data_start,
        access_profile
    FROM {DIM_CLIENTS}
    ORDER BY gsc_data_start NULLS LAST
    LIMIT 20
""").df()

print("Client history depth (first 20 clients, ordered by GSC start date):")
print(client_history.to_string())
print()
print("Clients with very recent gsc_data_start would have unreliable March feature values.")

---

## Self-check

Before submitting, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (all IDs are pseudonymized hashes)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Five plain-words contract answers in Section 1
- [x] Exactly three verification queries with outputs visible (grain, counts, availability IS TRUE)
- [x] Five-feature frame with an 'available when?' line per feature
- [x] Deliberate-leak experiment shown (imp_last15 added → precision spikes → deleted)
- [x] One named limitation stated and backed with a query
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.